<div style="display:none">
$$
\newcommand{\rvar}[1]{\mathrm{#1}}
\newcommand{\rvec}[1]{\mathbf{#1}}
\newcommand{\vec}[1]{\pmb{#1}}
\newcommand{\tens}[1]{\pmb{\mathsf{#1}}}
\newcommand{\tensel}[1]{\mathsf{#1}}
\newcommand{\st}[1]{\mathcal{#1}}
\newcommand{\diag}[1]{\mathrm{diag}(\vec{#1})}
$$
</div>

In [32]:
# Setup for Google Colab
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.exists('dl_course'):
        !git clone -q --depth 1 https://github.com/konstantin-schekotihin/dl_course.git
    %cd -q /content/dl_course/01_Introduction
    !pip install -q torch-geometric wandb

# some initialization stuff
%run ../init.py
%matplotlib inline

<IPython.core.display.Latex object>

<Figure size 1000x1000 with 0 Axes>

# Quantifying uncertainty

- Information theory founded by Claude Shannon to solve issues occurring while sending and receiving messages from discrete alphabet over noisy channels
  - The goal was to design specific en- and decoding methods allowing one to reduce the probability of an undetected errors to occur during the message exchange
  - Results obtained in the information theory are widely used in ML, see e.g. the book of MacKay

<div class="cite"> MacKay D.J.C.: Information theory, inference, and learning algorithms. Cambridge University Press 2003, ISBN 978-0-521-64298-9, pp. I-XII, 1-628</div>

## Example

- Assume we have two coins, a biased one with $P(\rvar{c}_1=H) = 0.75$ and a fair one $P(\rvar{c}_2=H) = 0.5$.  I toss the coins and ask you to guess the result. But you have a "joker", which allows you to ask a question about the outcome of one coin.
- Outcome of which coin is more informative for you?

- The key is in the joint probability distribution (tosses are independent)
  - $P(\rvar{c}_1=H,\rvar{c}_2=H) = 0.75\cdot 0.5=0.375$
  - $P(\rvar{c}_1=H,\rvar{c}_2=T) = 0.75\cdot 0.5=0.375$
  - $P(\rvar{c}_1=T,\rvar{c}_2=H) = 0.25\cdot 0.5=0.125$
  - $P(\rvar{c}_1=T,\rvar{c}_2=T) = 0.25\cdot 0.5=0.125$
  

- Consider conditional probabilities for the fair coin given the outcome of the unfair
  - $P(\rvar{c}_2=H | \rvar{c}_1=H) = 0.375/0.75 = 0.5$
  - $P(\rvar{c}_2=H | \rvar{c}_1=T) = 0.125/0.25 = 0.5$

- Now vice verse:

  $P(\rvar{c}_1=H | \rvar{c}_2=H) = 0.375/0.5 = 0.75$
  
  $P(\rvar{c}_1=T | \rvar{c}_2=H) = 0.125/0.5 = 0.25$


- So which outcome would you like to know?

## Entropy

- It would be nice in the previous example if we could somehow automate the decision procedure
  - Likely events should provide less information content than unlikely
  - Guaranteed events provide no information
  - Independent events should have additive information - outcomes of tossing two fair coins should provide twice as much information as tossing one fair coin 

- Self-information of an event $\rvar{x}=x$ (single outcome)
$$I(x)=-\log P(x)$$
where $\log$ denoted the natural logarithm.
  - In this case the information is measured in *nats* - amount of information gained by observing an outcome of an event with the probability $1/e$ 
  - In coding theory (binary signals) the $\log_2$ is more comfortable - measured in *bits*

## Shannon entropy

- Generalizes self-information to measure the uncertainty in an entire distribution

$$H(\text{x})=\mathbb{E}_{\rvar{x}\sim P}[I(x)]=-\mathbb{E}_{\rvar{x}\sim P}[\log P(x)]$$

- *Shannon entropy*: 
  - Quantifies the expected amount of information in an event drawn from $P$
  - Distributions that are almost deterministic have *low* entropy
  - Close to the uniform distributions have *high* entropy
  - $0 \log 0$ is treated as $\lim_{x\to 0} x\log x = 0$ 

- Consider we have a biased coin and a fair one. Which outcome we are more uncertain about?  

In [3]:
sp = np.linspace(0,1,101)
ent = lambda p : 0 if p==0 or p==1 else -((1-p)*np.log(1-p)+p*np.log(p))

@interact(x=(0,1,.10))
def draw_ent(x=0.5): 
    print('Entropy {:.2f} nits'.format(ent(x)))
    plt.plot(sp, list(map(ent, sp)), color="red", label="Entropy")
    plt.scatter(x, ent(x))
    plt.show();

interactive(children=(FloatSlider(value=0.5, description='x', max=1.0), Output()), _dom_classes=('widget-inter…

## Comparing the distributions

- Given two discrete probability distributions $P(\rvar{x})$ and $Q(\rvar{x})$ over the same random variable and sample space
- *Kullback-Leiber (KL) divergence* can be used to measure the difference between them

$$D_{KL}(P\parallel Q)=\mathbb{E}_{\rvar{x}\sim P}\left[\log\frac{P(x)}{Q(x)}\right]=\mathbb{E}_{\rvar{x}\sim P}\left[\log P(x) - \log Q(x)\right]$$

- Interesting properties:
    - KL divergence is non-negative
    - Is 0 iff $P$ and $Q$ are the same distribution
    - Can be seen as a distance between the distributions
    - In not symmetric $D_{KL}(P\parallel Q)\neq D_{KL}(Q\parallel P)$, so the choice of the measure is important

### KL Divergence: Example

- Given two vectors $P$ and $Q$
    - Check if they are disrtibutions
    - Check sample space 
    - Compute scores
    
        $$ 
            s_i = \left\{\begin{matrix}
                \ln(\frac{p_i}{q_i}) & p_i > 0, q_i > 0\\ 
                 0 & p_i = 0, q_i \geq 0\\ 
                 \infty & \mathit{otherwise}
                \end{matrix}\right. 
         $$
                       
    - Compute the expected value as $\sum_i p_i s_i $

In [30]:
def kld(p: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    for d in locals().values():
        if torch.sum(d) != 1 or d[(d<0)&(d>1)].nelement() > 0:
            raise Exception("P and Q must be distributions")
    if p.size() != q.size():
        raise Exception("P and Q must be distributions over the same random variable")

    res = torch.tensor(list(map(kld_entr, p, q)), dtype=torch.float)
    return torch.sum(p*res)


def kld_entr(p: torch.Tensor, q: torch.Tensor) -> float:
    if p > 0 and q > 0:
        return torch.log(p / q)
    elif p == 0 and q >= 0:
        return 0
    else:
        return float('inf')

In [31]:
# Multi-class classification problem with four classes

p  = torch.tensor([ 0,  1,  0,  0], dtype=torch.float)
q1 = torch.tensor([.1, .7, .1, .1])
q2 = torch.tensor([.7, .1, .1, .1])

print("KL diveregnce for P and Q1: ", kld(p, q1))
print("KL diveregnce for Q1 and P: ", kld(q1, p))
print("KL diveregnce for P and Q2: ", kld(p, q2))

KL diveregnce for P and Q1:  tensor(0.3567)
KL diveregnce for Q1 and P:  tensor(inf)
KL diveregnce for P and Q2:  tensor(2.3026)


## Other information-theoretic measures

- *Cross-entropy* is another measure closely related to KL divergence

$$H(P,Q)=H(P)+D_{KL}(P\parallel Q) = -\mathbb{E}_{\rvar{x}\sim P}[\log Q(x)]$$

- Derivation:
$$\begin{align}
H(P,Q) &= -\mathbb{E}_{\rvar{x}\sim P}\left[\log P(x)\right] + \mathbb{E}_{\rvar{x}\sim P}\left[\log P(x) - \log Q(x)\right] = \\
 &= \mathbb{E}_{\rvar{x}\sim P}\left[-\log P(x) + \log P(x) - \log Q(x)\right] \\ &= -\mathbb{E}_{\rvar{x}\sim P}[\log Q(x)]
 \end{align}$$

- Note, cross-entropy compares distributions over one variable $\rvar{x}$ and is different from the joint entropy $H(\text{x}_1,\dots,\text{x}_n)$ defined for $n$ random variables
- Minimizing the cross-entropy wrt. the distribution $Q$ is equivalent to minimizing the KL divergence

In [28]:
def ce(p: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    for d in locals().values():
        if torch.sum(d) != 1 or d[(d<0)&(d>1)].nelement() > 0:
            raise Exception("P and Q must be distributions")
    if p.size() != q.size():
        raise Exception("P and Q must be distributions over the same random variable")

    res = torch.tensor(list(map(ce_entr, p, q)), dtype=torch.float)
    return -1*torch.sum(p*res)


def ce_entr(p: torch.Tensor, q: torch.Tensor) -> float:
    if p > 0 and q > 0:
        return torch.log(q)
    elif p == 0 and q >= 0:
        return 0
    else:
        return float('inf')

In [29]:
print("Cross-entropy for P and Q1: ", ce(p, q1))
print("Cross-entropy for Q1 and P: ", ce(q1, p))
print("Cross-entropy for P and Q2: ", ce(p, q2))

Cross-entropy for P and Q1:  tensor(0.3567)
Cross-entropy for Q1 and P:  tensor(-inf)
Cross-entropy for P and Q2:  tensor(2.3026)


- Jensen–Shannon divergence (JSD)
  - used in Generative Adversarial Networks (GANs)
  - is symmetric and always has a finite value
  - JS distance is defined as a square root of JSD
  
$$\mathit{JSD}(P\parallel Q) = \frac{D_\mathit{KL}(P\parallel M)+D_\mathit{KL}(Q\parallel M)}{2}$$

$$M(x)=\frac{P(x)+Q(x)}{2}$$